In [ ]:
# Phase 0 (optional): auto-install dependencies, then re-run
# This notebook cannot embed a full Python environment. If packages are missing, this cell installs
# them from requirements.txt into the CURRENT kernel environment.
# After install, you usually must: Kernel -> Restart, then Run All again.

from pathlib import Path
import importlib.util
import subprocess
import sys

print("Python executable:", sys.executable)

REQ = Path("requirements.txt")
CORE = [
    "numpy",
    "pandas",
    "tifffile",
    "skimage",      # scikit-image
    "trackpy",
    "imageio",
    "torch",
    "cellpose",
]

missing = [m for m in CORE if importlib.util.find_spec(m) is None]

if missing:
    print("Missing packages:", missing)
    if not REQ.exists():
        raise RuntimeError(
            f"{REQ} not found. Put requirements.txt next to this notebook, or install dependencies manually."
        )

    print("Installing dependencies from:", REQ.resolve())
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ)])
    print("\nInstall finished. IMPORTANT: restart the kernel, then Run All again.")
else:
    print("Core dependencies already present. You can run Phase 1.")


In [ ]:
# Phase 1: Cellpose segmentation + centroid tracking (Python-only, reproducible)

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tifffile as tf

# IMPORTANT: Phase 1 must run in an environment with a working PyTorch + classic Cellpose install.
# (We validate capabilities below rather than relying on a specific venv folder name.)
print("Python executable:", sys.executable)

# --- Environment sanity checks (kernel-agnostic) ---
try:
    import torch  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "PyTorch failed to import in the current Jupyter kernel/environment.\n\n"
        "Fix:\n"
        "- Create/activate the project environment\n"
        "- Install deps (e.g. `pip install -r requirements.txt`)\n"
        "- In Jupyter, select that kernel and Restart Kernel\n\n"
        f"Original error importing torch: {e}"
    ) from e

try:
    from cellpose import models
except OSError as e:
    # Most common on Windows: torch DLL mismatch or missing runtime
    raise RuntimeError(
        "Failed to import cellpose (often due to PyTorch DLL issues).\n\n"
        "Fix:\n"
        "- Use an environment where PyTorch + Cellpose are installed correctly\n"
        "- Install deps (e.g. `pip install -r requirements.txt`)\n"
        "- Restart the kernel after switching environments\n\n"
        f"Original error: {e}"
    ) from e
except Exception as e:
    raise RuntimeError(
        "Failed to import cellpose in the current environment.\n\n"
        "Fix:\n"
        "- Install deps (e.g. `pip install -r requirements.txt`)\n"
        "- Restart the kernel\n\n"
        f"Original error: {e}"
    ) from e

from skimage.measure import regionprops_table
from skimage.segmentation import relabel_sequential
import trackpy as tp

# This phase expects classic Cellpose (cyto2), i.e. cellpose < 4.
if not hasattr(models, "Cellpose"):
    raise ImportError(
        "This notebook expects classic Cellpose (cellpose<4), i.e. models.Cellpose is available. "
        "Install cellpose<4 (cyto2 API) in your environment, then restart the kernel and rerun."
    )

root = Path(r"C:\Users\fgkb\Desktop\bioelectricity project")
ctrl_path = root / "Ctrl_0mV_5min_interval-2.tif"
ef_path = root / "EF_200mV_5min_interval-2.tif"

# Calibration from Kan
DT_MIN = 5
PIXELS_PER_UM = 1.135
UM_PER_PX = 1.0 / PIXELS_PER_UM

# Kan: EF was applied after the first frame (frame 0 baseline)
EF_ON_FRAME = 1
EF_ON_TIME_MIN = EF_ON_FRAME * DT_MIN

# Convention: cathode is LEFT on screen, so leftward is positive along EF axis.
EF_AXIS = "x"   # 'x' or 'y'
EF_SIGN = -1    # leftward positive

# Cellpose parameters (start here and tune)
MODEL_TYPE = "cyto2"
DIAMETER_PX = 30  # starting guess
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0

# Automatic per-frame QC filter (applied BEFORE tracking)
# Tune these after you run full 37 frames and inspect failures.
QC_MIN_AREA_PX = 200
QC_MAX_AREA_PX = 6000
# Start looser for phase-contrast; tighten after you inspect distributions/time stability.
QC_MIN_SOLIDITY = 0.80
QC_MIN_ECCENTRICITY = 0.15
QC_MAX_CIRCULARITY = 0.90  # lower removes very circular/ring-like artefacts
QC_BORDER_PX = 8  # exclude detections too close to image border
QC_VERBOSE = False

# Tracking parameters
SEARCH_RANGE_PX = 20   # linking max distance between frames
MEMORY = 1             # allow 1 missing frame


def run_cellpose_on_stack(path: Path, max_frames: int | None = None):
    """Run Cellpose per-frame to avoid loading entire stack into RAM."""
    mm = tf.memmap(str(path))  # (T, Y, X) uint16
    nT = int(mm.shape[0])
    T = nT if max_frames is None else min(nT, int(max_frames))

    model = models.Cellpose(gpu=False, model_type=MODEL_TYPE)

    masks = []
    for t in range(T):
        img = mm[t]
        # Classic Cellpose API: use channels=[0, 0] for single-channel 2D images
        m, flows, styles, diams = model.eval(
            img,
            channels=[0, 0],
            diameter=DIAMETER_PX,
            flow_threshold=FLOW_THRESHOLD,
            cellprob_threshold=CELLPROB_THRESHOLD,
            normalize=True,
        )
        masks.append(m.astype(np.int32, copy=False))

    return np.stack(masks, axis=0)  # (T, Y, X)


def centroids_from_masks(masks: np.ndarray) -> tuple[pd.DataFrame, np.ndarray]:
    """Extract centroids + region features and apply a simple QC filter per frame.

    Returns:
      - pts_df: DataFrame with columns frame,x,y,label (+ features)
      - masks_filt: relabeled masks after QC filtering (same shape as input)
    """

    rows = []
    masks_filt = np.zeros_like(masks, dtype=np.int32)

    for t in range(masks.shape[0]):
        m = masks[t]
        if int(m.max()) == 0:
            continue

        props = regionprops_table(
            m,
            properties=(
                "label",
                "centroid",
                "area",
                "eccentricity",
                "solidity",
                "perimeter",
            ),
        )
        df = pd.DataFrame(props)
        df.rename(columns={"centroid-0": "y", "centroid-1": "x"}, inplace=True)
        df["frame"] = t

        # Circularity: 4Ď€A / P^2 (1=circle). Avoid divide-by-zero.
        per = df["perimeter"].to_numpy(dtype=float)
        area = df["area"].to_numpy(dtype=float)
        circ = np.zeros_like(area)
        ok = per > 0
        circ[ok] = (4.0 * np.pi * area[ok]) / (per[ok] ** 2)
        df["circularity"] = circ

        H, W = m.shape
        keep = (
            (df["area"] >= QC_MIN_AREA_PX)
            & (df["area"] <= QC_MAX_AREA_PX)
            & (df["solidity"] >= QC_MIN_SOLIDITY)
            & (df["eccentricity"] >= QC_MIN_ECCENTRICITY)
            & (df["circularity"] <= QC_MAX_CIRCULARITY)
            & (df["x"] >= QC_BORDER_PX)
            & (df["x"] <= (W - 1 - QC_BORDER_PX))
            & (df["y"] >= QC_BORDER_PX)
            & (df["y"] <= (H - 1 - QC_BORDER_PX))
        )
        df_keep = df[keep].copy()

        if QC_VERBOSE:
            print(
                f"frame {t}: total={len(df):4d} kept={len(df_keep):4d} removed={len(df)-len(df_keep):4d}"
            )

        # Filter mask and relabel sequentially for clean viewing/tracking
        kept_labels = df_keep["label"].to_numpy(dtype=int)
        mf = np.where(np.isin(m, kept_labels), m, 0).astype(np.int32, copy=False)
        mf, fw, _ = relabel_sequential(mf)
        masks_filt[t] = mf

        # Store centroid rows (use NEW labels after relabeling)
        # fw maps old_label -> new_label (skimage returns an ArrayMap; index it like fw[old_label])
        for _, r in df_keep.iterrows():
            old_label = int(r["label"])
            new_label = int(fw[old_label])
            rows.append(
                {
                    "frame": int(r["frame"]),
                    "label": new_label,
                    "label_old": old_label,
                    "x": float(r["x"]),
                    "y": float(r["y"]),
                    "area": float(r["area"]),
                    "eccentricity": float(r["eccentricity"]),
                    "solidity": float(r["solidity"]),
                    "circularity": float(r["circularity"]),
                }
            )

    pts_df = pd.DataFrame(rows)
    return pts_df, masks_filt


JUMP_MAX_FACTOR = 2.5  # "insane jump" threshold relative to SEARCH_RANGE_PX


def track_centroids(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    # Avoid passing label fields into the tracker (label != identity)
    df_track = df[["frame", "x", "y"]].copy()

    linked = tp.link_df(df_track, search_range=SEARCH_RANGE_PX, memory=MEMORY)

    # Split tracks on insane jumps (or non-consecutive frames) to reduce identity-swap pollution
    linked = linked.sort_values(["particle", "frame"]).copy()
    linked["step"] = linked.groupby("particle")["frame"].diff()
    linked["dx_raw"] = linked.groupby("particle")["x"].diff()
    linked["dy_raw"] = linked.groupby("particle")["y"].diff()
    linked["jump_px"] = np.hypot(linked["dx_raw"], linked["dy_raw"])

    jump_max = float(JUMP_MAX_FACTOR) * float(SEARCH_RANGE_PX)
    linked["break"] = (linked["step"] != 1) | (linked["jump_px"] > jump_max)
    linked["segment"] = linked.groupby("particle")["break"].cumsum().fillna(0).astype(int)

    linked["particle"] = (linked["particle"].astype(int) * 10000 + linked["segment"]).astype(int)
    linked.drop(columns=["segment"], inplace=True)

    # Track length summary
    counts = linked.groupby("particle").size()
    print(
        f"tracks: n_particles={counts.shape[0]} | mean_len={counts.mean():.2f} | min_len={counts.min()} | max_len={counts.max()} | jump_max_px={jump_max:.1f}"
    )

    # Filter short tracks
    keep = counts[counts >= MIN_TRACK_LEN].index
    linked = linked[linked["particle"].isin(keep)].copy()
    print(
        f"after filtering (>= {MIN_TRACK_LEN} frames): n_rows={linked.shape[0]} n_particles={linked['particle'].nunique() if not linked.empty else 0}"
    )

    return linked


def drift_correct_steps(tracks: pd.DataFrame) -> pd.DataFrame:
    """Subtract per-frame median displacement across all particles (stage drift correction).

    dx/dy at frame=t represent motion from (t-1 -> t).
    """

    tracks = tracks.sort_values(["particle", "frame"]).copy()
    tracks[["dx", "dy"]] = tracks.groupby("particle")[["x", "y"]].diff()

    # Compute drift only from valid steps (drop NaNs from the first row of each particle)
    step_rows = tracks.dropna(subset=["dx", "dy"]).copy()
    drift = (
        step_rows.groupby("frame")[["dx", "dy"]]
        .median()
        .rename(columns={"dx": "drift_dx", "dy": "drift_dy"})
    )
    tracks = tracks.join(drift, on="frame")

    tracks["dx_corr"] = tracks["dx"] - tracks["drift_dx"]
    tracks["dy_corr"] = tracks["dy"] - tracks["drift_dy"]
    return tracks


def directed_component_um_per_min(tracks: pd.DataFrame) -> pd.Series:
    dx_col = "dx_corr" if "dx_corr" in tracks.columns else "dx"
    dy_col = "dy_corr" if "dy_corr" in tracks.columns else "dy"
    comp = tracks[dx_col] if EF_AXIS == "x" else tracks[dy_col]
    # px/frame -> Âµm/min
    return (EF_SIGN * comp) * UM_PER_PX / DT_MIN


# --- Run segmentation (start with a few frames to sanity check) ---
# Set to e.g. 5 to test quickly, then None for full run.
TEST_FRAMES = None

# IMPORTANT: if you only run a few frames, don't require long tracks.
# With TEST_FRAMES=5, the maximum possible track length is 5.
MIN_TRACK_LEN = (max(3, TEST_FRAMES - 1) if TEST_FRAMES is not None else 10)
print("MIN_TRACK_LEN:", MIN_TRACK_LEN)

print("Running Cellpose (CTRL test frames)...")
ctrl_masks = run_cellpose_on_stack(ctrl_path, max_frames=TEST_FRAMES)
print("Running Cellpose (EF test frames)...")
ef_masks = run_cellpose_on_stack(ef_path, max_frames=TEST_FRAMES)

print("CTRL masks shape:", ctrl_masks.shape, "labels max:", int(ctrl_masks.max()))
print("EF   masks shape:", ef_masks.shape, "labels max:", int(ef_masks.max()))

# --- Convert to centroids (with QC filter) and track ---
ctrl_pts, ctrl_masks_filt = centroids_from_masks(ctrl_masks)
ef_pts, ef_masks_filt = centroids_from_masks(ef_masks)

print("CTRL centroids kept:", len(ctrl_pts))
print("EF   centroids kept:", len(ef_pts))

ctrl_tracks = track_centroids(ctrl_pts)
ef_tracks = track_centroids(ef_pts)

if APPLY_DRIFT_CORRECTION:
    ctrl_tracks = drift_correct_steps(ctrl_tracks) if not ctrl_tracks.empty else ctrl_tracks
    ef_tracks = drift_correct_steps(ef_tracks) if not ef_tracks.empty else ef_tracks
else:
    for _name, _df in [("ctrl", ctrl_tracks), ("ef", ef_tracks)]:
        if not _df.empty:
            _df = _df.sort_values(["particle", "frame"]).copy()
            _df[["dx", "dy"]] = _df.groupby("particle")[["x", "y"]].diff()
            if _name == "ctrl":
                ctrl_tracks = _df
            else:
                ef_tracks = _df
    print("Drift correction SKIPPED (APPLY_DRIFT_CORRECTION=False)")

# Since EF was applied after frame 0, focus analysis on steps ending at frame >= EF_ON_FRAME.
# Note: dx/dy is the step from (t-1 -> t), and the first step is at frame=1.
STEP_START_FRAME = max(EF_ON_FRAME, 1)

if not ctrl_tracks.empty:
    ctrl_v_post = directed_component_um_per_min(ctrl_tracks[ctrl_tracks["frame"] >= STEP_START_FRAME]).dropna()
else:
    ctrl_v_post = pd.Series(dtype=float)

if not ef_tracks.empty:
    ef_v_post = directed_component_um_per_min(ef_tracks[ef_tracks["frame"] >= STEP_START_FRAME]).dropna()
else:
    ef_v_post = pd.Series(dtype=float)

print(f"EF ON at frame {EF_ON_FRAME} (t={EF_ON_TIME_MIN} min)")
print(f"Analyzing steps ending at frame >= {STEP_START_FRAME}")
print(
    "CTRL directed v post (Âµm/min): n=",
    len(ctrl_v_post),
    " meanÂ±std=",
    float(ctrl_v_post.mean()) if len(ctrl_v_post) else None,
    "Â±",
    float(ctrl_v_post.std()) if len(ctrl_v_post) else None,
)
print(
    "EF   directed v post (Âµm/min): n=",
    len(ef_v_post),
    " meanÂ±std=",
    float(ef_v_post.mean()) if len(ef_v_post) else None,
    "Â±",
    float(ef_v_post.std()) if len(ef_v_post) else None,
)

# --- Per-cell outcomes (more standard electrotaxis plots) ---

def per_cell_metrics(tracks: pd.DataFrame) -> pd.DataFrame:
    if tracks.empty:
        return pd.DataFrame()

    dc = "dx_corr" if "dx_corr" in tracks.columns else "dx"
    dy_col = "dy_corr" if "dy_corr" in tracks.columns else "dy"
    steps = tracks[(tracks["frame"] >= STEP_START_FRAME)].dropna(subset=[dc, dy_col]).copy()
    if steps.empty:
        return pd.DataFrame()

    g = steps.groupby("particle")

    ef_comp = steps[dc] if EF_AXIS == "x" else steps[dy_col]
    sum_ef = ef_comp.groupby(steps["particle"]).sum()
    sum_dx = g[dc].sum()
    sum_dy = g[dy_col].sum()
    step_dist = np.sqrt(steps[dc] ** 2 + steps[dy_col] ** 2)
    path_len = step_dist.groupby(steps["particle"]).sum()
    net_disp = np.sqrt(sum_dx ** 2 + sum_dy ** 2)

    out = pd.DataFrame({"n_steps": g.size()})

    # Old path-based directedness (kept for comparison)
    out["directedness_path"] = (EF_SIGN * sum_ef) / path_len.replace(0, np.nan)

    # Paper directedness: cos(theta) = net EF-axis / net straight-line displacement
    # (Zhang et al., iScience 2022)
    out["directedness_cos"] = (EF_SIGN * sum_ef) / net_disp.replace(0, np.nan)

    out["net_toward_cathode_um"] = (EF_SIGN * sum_ef) * UM_PER_PX
    out["net_displacement_um"] = net_disp * UM_PER_PX

    ef_step_vel = (EF_SIGN * ef_comp) * UM_PER_PX / DT_MIN
    step_speed = step_dist * UM_PER_PX / DT_MIN

    out["avg_step_speed_um_per_min"] = step_speed.groupby(steps["particle"]).mean()
    out["avg_speed_toward_cathode_um_per_min"] = ef_step_vel.groupby(steps["particle"]).mean()

    out = out.dropna(subset=["directedness_cos"])
    return out


ctrl_cell = per_cell_metrics(ctrl_tracks)
ef_cell = per_cell_metrics(ef_tracks)

print("Per-cell metrics (post-EF):")
print("  CTRL n_cells:", int(ctrl_cell.shape[0]))
print("  EF   n_cells:", int(ef_cell.shape[0]))

# Export per-cell CSV
out_csv_dir = root / "results" / "csv_exports"
out_csv_dir.mkdir(parents=True, exist_ok=True)
ctrl_cell.to_csv(out_csv_dir / "ctrl_per_cell.csv", index=True)
ef_cell.to_csv(out_csv_dir / "ef_per_cell.csv", index=True)
print(f"Per-cell CSVs saved to {out_csv_dir}")

# Export full centroid tracking CSV (all frames, all particles)
ctrl_tracks.to_csv(out_csv_dir / "ctrl_tracks_full.csv", index=False)
ef_tracks.to_csv(out_csv_dir / "ef_tracks_full.csv", index=False)
print(f"Full tracks CSVs saved to {out_csv_dir}")

if TEST_FRAMES is not None:
    print("Tip: set TEST_FRAMES=None once masks look good to run full 37 frames.")


Frame 36: 79 trajectories present.


tracks: n_particles=239 | mean_len=10.80 | min_len=1 | max_len=37 | jump_max_px=50.0
after filtering (>= 10 frames): n_rows=2141 n_particles=81
EF ON at frame 1 (t=5 min)
Analyzing steps ending at frame >= 1
CTRL directed v post (µm/min): n= 2269  mean±std= -0.02708696691485547 ± 1.0855487904391217
EF   directed v post (µm/min): n= 2060  mean±std= 0.0028903342321233683 ± 0.3807191581407619
Per-cell metrics (post-EF):
  CTRL n_cells: 118
  EF   n_cells: 81
Tip: set TEST_FRAMES=None once masks look good to run full 37 frames.


In [2]:
# QC STEP 2: visualize picked centroids (and tracks) in napari
# Run this AFTER Phase 1 so ef_pts/ctrl_pts and ef_tracks/ctrl_tracks exist.

import napari

VIEW_MODE = "both"  # "ef", "ctrl", or "both"
FRAME_TO_SHOW = 4

# For TEST_FRAMES runs, masks/pts only cover the first T frames.
if VIEW_MODE == "both":
    T = int(min(ctrl_masks_filt.shape[0], ef_masks_filt.shape[0]))
    ctrl_vol = tf.memmap(str(ctrl_path))[:T]
    ef_vol = tf.memmap(str(ef_path))[:T]

    # Side-by-side: LEFT=ctrl, RIGHT=EF
    X = int(ctrl_vol.shape[2])
    vol = np.concatenate([ctrl_vol, ef_vol], axis=2)

    # Avoid label id collisions by offsetting EF labels per-frame
    masks = np.zeros_like(vol, dtype=np.int32)
    pts = None
    tracks = None

    for t in range(T):
        cm = ctrl_masks_filt[t]
        em = ef_masks_filt[t]
        off = int(cm.max())
        masks[t, :, :X] = cm
        masks[t, :, X:] = np.where(em > 0, em + off, 0)

    # Points: concatenate with x-offset for EF
    ctrl_coords = ctrl_pts[["frame", "y", "x"]].to_numpy(dtype=float)
    ef_coords = ef_pts[["frame", "y", "x"]].to_numpy(dtype=float)
    ef_coords[:, 2] += X
    centroid_coords = np.vstack([ctrl_coords, ef_coords])

    # Tracks: offset EF particle ids to avoid collisions
    if (ctrl_tracks is not None and (not ctrl_tracks.empty)) or (ef_tracks is not None and (not ef_tracks.empty)):
        ctd = ctrl_tracks[["particle", "frame", "y", "x"]].to_numpy(dtype=float) if (ctrl_tracks is not None and (not ctrl_tracks.empty)) else np.zeros((0, 4))
        etd = ef_tracks[["particle", "frame", "y", "x"]].to_numpy(dtype=float) if (ef_tracks is not None and (not ef_tracks.empty)) else np.zeros((0, 4))
        if etd.shape[0] > 0:
            etd[:, 0] += (ctd[:, 0].max() + 1) if ctd.shape[0] > 0 else 0
            etd[:, 3] += X
        track_data = np.vstack([ctd, etd]) if (ctd.shape[0] + etd.shape[0]) > 0 else None
    else:
        track_data = None

    title = "CTRL (LEFT) | EF (RIGHT) centroids QC"

elif VIEW_MODE == "ef":
    T = int(ef_masks_filt.shape[0])
    vol = tf.memmap(str(ef_path))[:T]
    masks = ef_masks_filt
    centroid_coords = ef_pts[["frame", "y", "x"]].to_numpy(dtype=float)
    track_data = ef_tracks[["particle", "frame", "y", "x"]].to_numpy(dtype=float) if (ef_tracks is not None and (not ef_tracks.empty)) else None
    title = "EF centroids QC"

else:
    T = int(ctrl_masks_filt.shape[0])
    vol = tf.memmap(str(ctrl_path))[:T]
    masks = ctrl_masks_filt
    centroid_coords = ctrl_pts[["frame", "y", "x"]].to_numpy(dtype=float)
    track_data = ctrl_tracks[["particle", "frame", "y", "x"]].to_numpy(dtype=float) if (ctrl_tracks is not None and (not ctrl_tracks.empty)) else None
    title = "CTRL centroids QC"

viewer = napari.Viewer(title=title)
# QC visualization: use SPATIAL scaling only (T is an index, not a physical dimension here)
QC_SCALE = (1.0, UM_PER_PX, UM_PER_PX)

viewer.add_image(
    vol,
    name="stack",
    scale=QC_SCALE,
    colormap="gray",
)
viewer.add_labels(
    masks,
    name="masks_filt",
    opacity=0.35,
    scale=QC_SCALE,
)

viewer.add_points(
    centroid_coords,
    name="centroids",
    size=6 * UM_PER_PX,  # ~6 px expressed in µm
    scale=QC_SCALE,
    face_color="cyan",
    edge_color="black",
    opacity=0.9,
)

if track_data is not None and track_data.shape[0] > 0:
    viewer.add_tracks(
        track_data,
        name="tracks",
        scale=QC_SCALE,
        tail_length=20,
        blending="translucent_no_depth",
    )

viewer.dims.set_point(0, min(FRAME_TO_SHOW, T - 1))
napari.run()


In [ ]:
# Export: segmentation overlay video (CTRL left | EF right)
# Run AFTER Phase 2 so ctrl_masks_filt/ef_masks_filt exist.

from skimage.segmentation import find_boundaries
import imageio.v2 as imageio

#root = Path(r"C:\Users\fgkb\Desktop\bioelectricity project")
#ctrl_path = root / "Ctrl_0mV_5min_interval-2.tif"
#ef_path = root / "EF_200mV_5min_interval-2.tif"

out_dir = root / "results" / "qc_videos"
out_dir.mkdir(parents=True, exist_ok=True)

OUT_PATH = out_dir / "segmentation_overlay_ctrl_left__ef_right.mp4"
FPS = 5  # playback speed (not real time)

# Kan: EF was applied after the first frame (frame 0 baseline)
EF_ON_FRAME = 1


def _compute_vmin_vmax(mm: np.ndarray, step: int = 5):
    # sample every `step` frames to reduce work
    sample = np.asarray(mm[::step])
    vmin, vmax = np.percentile(sample, (1, 99))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)


def _to_uint8(img16: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    x = img16.astype(np.float32)
    x = (x - vmin) / (vmax - vmin)
    x = np.clip(x, 0.0, 1.0)
    return (x * 255).astype(np.uint8)


def _overlay_boundaries(gray_u8: np.ndarray, mask: np.ndarray, color_rgb=(0, 255, 0)) -> np.ndarray:
    rgb = np.stack([gray_u8, gray_u8, gray_u8], axis=-1)
    b = find_boundaries(mask > 0, mode="outer")
    rgb[b, 0] = color_rgb[0]
    rgb[b, 1] = color_rgb[1]
    rgb[b, 2] = color_rgb[2]
    return rgb


# Load movies lazily
ctrl_mm = tf.memmap(str(ctrl_path))  # (T,Y,X)
ef_mm = tf.memmap(str(ef_path))

T = int(min(ctrl_masks_filt.shape[0], ef_masks_filt.shape[0], ctrl_mm.shape[0], ef_mm.shape[0]))
H = int(ctrl_mm.shape[1])
X = int(ctrl_mm.shape[2])

ctrl_vmin, ctrl_vmax = _compute_vmin_vmax(ctrl_mm)
ef_vmin, ef_vmax = _compute_vmin_vmax(ef_mm)

print("Writing:", OUT_PATH)
print("frames:", T, " size:", (H, X * 2), " fps:", FPS)

# macro_block_size=1 avoids ffmpeg errors on non-multiple-of-16 frame sizes
with imageio.get_writer(
    str(OUT_PATH),
    fps=FPS,
    codec="libx264",
    format="FFMPEG",
    macro_block_size=1,
    pixelformat="yuv420p",
) as w:
    for t in range(T):
        g1 = _to_uint8(ctrl_mm[t], ctrl_vmin, ctrl_vmax)
        g2 = _to_uint8(ef_mm[t], ef_vmin, ef_vmax)

        left = _overlay_boundaries(g1, ctrl_masks_filt[t], color_rgb=(0, 255, 0))      # green
        right = _overlay_boundaries(g2, ef_masks_filt[t], color_rgb=(255, 0, 255))     # magenta

        frame = np.concatenate([left, right], axis=1)

        # Simple EF-on indicator: add a small red square in the top-left corner of EF panel after EF_ON_FRAME
        if t >= EF_ON_FRAME:
            y0, y1 = 5, 25
            x0, x1 = X + 5, X + 25
            frame[y0:y1, x0:x1, :] = (255, 0, 0)

        w.append_data(frame)

print("Done.")
print("Tip: share this MP4 with the team:", OUT_PATH)


Writing: C:\Users\fgkb\Desktop\bioelectricity project\results\qc_videos\segmentation_overlay_ctrl_left__ef_right.mp4
frames: 37  size: (846, 2292)  fps: 5
Done.
Tip: share this MP4 with the team: C:\Users\fgkb\Desktop\bioelectricity project\results\qc_videos\segmentation_overlay_ctrl_left__ef_right.mp4


In [ ]:
# Export: masks + centroids + motion tracks video (CTRL left | EF right)
# Run AFTER Phase 2 so ctrl_masks_filt/ef_masks_filt and ctrl_tracks/ef_tracks exist.

import cv2
from skimage.color import label2rgb

root = Path(r"C:\Users\fgkb\Desktop\bioelectricity project")
out_dir = root / "results" / "qc_videos"
out_dir.mkdir(parents=True, exist_ok=True)

OUT_PATH = out_dir / "tracking_overlay_ctrl_left__ef_right.mp4"
FPS = 5          # playback speed (not real time)
TAIL_FRAMES = 10 # how many previous frames to draw as a tail
SHOW_IDS = True  # draw particle id text at track head

EF_ON_FRAME = 1


def _compute_vmin_vmax(mm: np.ndarray, step: int = 5):
    sample = np.asarray(mm[::step])
    vmin, vmax = np.percentile(sample, (1, 99))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)


def _to_float01(img16: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    x = img16.astype(np.float32)
    x = (x - vmin) / (vmax - vmin)
    return np.clip(x, 0.0, 1.0)


def _mask_overlay(gray01: np.ndarray, labels: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    # returns float RGB 0..1
    return label2rgb(labels, image=gray01, bg_label=0, alpha=alpha, image_alpha=1.0)


def _draw_centroids(
    rgb_u8: np.ndarray,
    pts_df,
    t: int,
    x_offset: int = 0,
    color=(255, 255, 0),  # RGB
):
    this = pts_df[pts_df["frame"] == t]
    for _, r in this.iterrows():
        x = int(round(float(r["x"]) + x_offset))
        y = int(round(float(r["y"])) )
        cv2.circle(rgb_u8, (x, y), 4, color, thickness=-1, lineType=cv2.LINE_AA)
        cv2.circle(rgb_u8, (x, y), 4, (0, 0, 0), thickness=1, lineType=cv2.LINE_AA)


def _draw_track_tails(rgb_u8: np.ndarray, tracks_df, t: int, x_offset: int = 0):
    # Draw tails for all particles up to frame t (last TAIL_FRAMES)
    if tracks_df is None or tracks_df.empty:
        return

    t0 = max(0, t - TAIL_FRAMES)
    sub = tracks_df[(tracks_df["frame"] >= t0) & (tracks_df["frame"] <= t)].copy()
    if sub.empty:
        return

    # color by particle id using a simple hash -> HSV -> RGB
    for pid, g in sub.groupby("particle"):
        g = g.sort_values("frame")
        pts = [
            (int(round(x_offset + x)), int(round(y)))
            for x, y in zip(g["x"], g["y"])
            if np.isfinite(x) and np.isfinite(y)
        ]
        if len(pts) < 2:
            continue

        h = int(pid) % 180
        col = cv2.cvtColor(np.uint8([[[h, 200, 255]]]), cv2.COLOR_HSV2RGB)[0, 0].tolist()
        col = (int(col[0]), int(col[1]), int(col[2]))

        for (x1, y1), (x2, y2) in zip(pts[:-1], pts[1:]):
            cv2.line(rgb_u8, (x1, y1), (x2, y2), col, thickness=2, lineType=cv2.LINE_AA)


def _draw_track_ids(rgb_u8: np.ndarray, tracks_df, t: int, x_offset: int = 0):
    if (not SHOW_IDS) or tracks_df is None or tracks_df.empty:
        return

    sub = tracks_df[tracks_df["frame"] == t]
    if sub.empty:
        return

    for _, r in sub.iterrows():
        pid = int(r["particle"])
        x = int(round(float(r["x"]) + x_offset))
        y = int(round(float(r["y"])) )
        h = pid % 180
        col = cv2.cvtColor(np.uint8([[[h, 200, 255]]]), cv2.COLOR_HSV2RGB)[0, 0].tolist()
        col = (int(col[0]), int(col[1]), int(col[2]))

        cv2.putText(
            rgb_u8,
            str(pid),
            (x + 6, y - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (0, 0, 0),
            thickness=2,
            lineType=cv2.LINE_AA,
        )
        cv2.putText(
            rgb_u8,
            str(pid),
            (x + 6, y - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            col,
            thickness=1,
            lineType=cv2.LINE_AA,
        )


# Load movies lazily
ctrl_mm = tf.memmap(str(ctrl_path))  # (T,Y,X)
ef_mm = tf.memmap(str(ef_path))

T = int(min(ctrl_masks_filt.shape[0], ef_masks_filt.shape[0], ctrl_mm.shape[0], ef_mm.shape[0]))
H = int(ctrl_mm.shape[1])
W = int(ctrl_mm.shape[2])

ctrl_vmin, ctrl_vmax = _compute_vmin_vmax(ctrl_mm)
ef_vmin, ef_vmax = _compute_vmin_vmax(ef_mm)

print("Writing:", OUT_PATH)
print("frames:", T, " size:", (H, W * 2), " fps:", FPS, " tail_frames:", TAIL_FRAMES)

with imageio.get_writer(str(OUT_PATH), fps=FPS) as w:
    for t in range(T):
        g1 = _to_float01(ctrl_mm[t], ctrl_vmin, ctrl_vmax)
        g2 = _to_float01(ef_mm[t], ef_vmin, ef_vmax)

        left = _mask_overlay(g1, ctrl_masks_filt[t], alpha=0.35)
        right = _mask_overlay(g2, ef_masks_filt[t], alpha=0.35)

        frame01 = np.concatenate([left, right], axis=1)  # float 0..1 RGB
        frame = (frame01 * 255).astype(np.uint8)

        # Draw track tails first, then centroids + IDs on top
        _draw_track_tails(frame, ctrl_tracks, t, x_offset=0)
        _draw_track_tails(frame, ef_tracks, t, x_offset=W)

        _draw_centroids(frame, ctrl_pts, t, x_offset=0)
        _draw_centroids(frame, ef_pts, t, x_offset=W)

        _draw_track_ids(frame, ctrl_tracks, t, x_offset=0)
        _draw_track_ids(frame, ef_tracks, t, x_offset=W)

        # EF-on indicator: red square in EF panel
        if t >= EF_ON_FRAME:
            y0, y1 = 5, 25
            x0, x1 = W + 5, W + 25
            frame[y0:y1, x0:x1, :] = (255, 0, 0)

        w.append_data(frame)

print("Done.")
print("Share this MP4:", OUT_PATH)


Writing: C:\Users\fgkb\Desktop\bioelectricity project\results\qc_videos\tracking_overlay_ctrl_left__ef_right.mp4
frames: 37  size: (846, 2292)  fps: 5  tail_frames: 10


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2292, 846) to (2304, 848) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Done.
Share this MP4: C:\Users\fgkb\Desktop\bioelectricity project\results\qc_videos\tracking_overlay_ctrl_left__ef_right.mp4
